# Huấn luyện DUSN-X state updater và router trên Google Colab
Huấn luyện mạng nơ-ron recurrent state updater và 3 đầu phân loại intent/agent/action; **không fine-tune LLM**. Chạy từng ô tuần tự từ trên xuống dưới.
- Checkpoint, metrics, logs, data manifest và báo cáo Markdown đều được lưu tự động vào Google Drive (`MyDrive/DUSNX/<RUN_NAME>`).
- Nếu runtime bị ngắt, đặt `RESUME = True` và chạy lại để tiếp tục từ `router.last.pt`.
- Hỗ trợ smoke CPU (kiểm tra 2-3 phút) và full GPU training (yêu cầu GPU T4/V100/A100 trên Colab).
- Tập đánh giá `benchmarks/holdout_v2.jsonl` đã khóa hash, đánh giá trung thực tách biệt 5 cấu hình: `baseline_a` (no memory), `baseline_b` (static memory), `model_only` (pure checkpoint, không luật), `dusnx_no_state` (ablation không recurrent state), và `dusnx` (full system).

In [ ]:
import sys, subprocess, pathlib, os
assert (3, 11) <= sys.version_info[:2] <= (3, 14), "Cần Python 3.11–3.14"
PROJECT = pathlib.Path("/content/dusnx-platform")
REPOSITORY = "https://github.com/quocdoanhdang2-sketch/dusnx-platform.git"
REVISION = "main"  # Đổi thành commit SHA cụ thể nếu muốn tái lập chính xác một mốc.
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(PROJECT)], check=True)
    subprocess.run(["git", "checkout", REVISION], cwd=PROJECT, check=True)
else:
    subprocess.run(["git", "fetch", "origin"], cwd=PROJECT, check=True)
    subprocess.run(["git", "checkout", REVISION], cwd=PROJECT, check=True)
os.chdir(PROJECT)
commit_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository HEAD:", commit_sha)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "./python[dev,data]"], check=True)
sys.path[:0] = [str(PROJECT / "python/src"), str(PROJECT / "python"), str(PROJECT / "python/scripts")]
os.environ["PYTHONPATH"] = os.pathsep.join(sys.path[:3])


In [ ]:
from colab_helper import check_environment
# Đặt SMOKE = True để kiểm tra nhanh pipeline trên CPU (2-3 phút).
# Đặt SMOKE = False khi đã chọn GPU (Runtime > Change runtime type > GPU) để chạy full training.
SMOKE = True
env_info = check_environment(large=not SMOKE)
print("Thông tin môi trường thực thi:", env_info)


## Kết nối Google Drive
Bấm Connect to Google Drive, chọn tài khoản và cấp quyền. Mỗi lần chạy mới đặt `RUN_NAME` khác để không ghi đè checkpoint; nếu muốn resume từ lần chạy trước, giữ nguyên `RUN_NAME` cũ.

In [ ]:
import pathlib
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = pathlib.Path("/content/drive/MyDrive/DUSNX")
except ImportError:
    DRIVE_DIR = pathlib.Path("runtime/drive_mock/DUSNX")
RUN_NAME = "dusnx-router-v2"
OUTPUT = DRIVE_DIR / RUN_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Thư mục lưu trữ kết quả trên Drive:", OUTPUT)


In [ ]:
# Smoke không cần tải dataset ngoài. Full train tải nguồn public theo manifest cố định.
if not SMOKE:
    subprocess.run([sys.executable, "python/scripts/fetch_sources.py", "--source", "all"], check=True)
    subprocess.run([sys.executable, "python/scripts/import_data.py", "--source", "massive",
                    "--input", "runtime/external-data/massive/vi-VN.jsonl",
                    "--output-dir", "runtime/imported/massive"], check=True)
# CSConDa là tập gated; mặc định bỏ qua nếu chưa được cấp quyền trên Hugging Face.
# Nếu đã được cấp quyền, lưu HF_TOKEN vào Colab Secrets rồi chạy: fetch_sources.py --source csconda --allow-gated.


In [ ]:
import json, shutil
from prepare_data import prepare
LEGACY = DRIVE_DIR / "input/synthetic_30k_v2.jsonl"
BASELINE = DRIVE_DIR / "input/dusnx_smoke_v2.pt"
external = PROJECT / "runtime/imported/massive/candidates.jsonl"
legacy_path = LEGACY if LEGACY.exists() else (PROJECT / "data/synthetic_30k_v2.jsonl" if (PROJECT / "data/synthetic_30k_v2.jsonl").exists() else None)
report = prepare(PROJECT / "runtime/prepared", legacy=legacy_path,
                 external=external if external.exists() and not SMOKE else None)
shutil.copy2(PROJECT / "runtime/prepared/manifest.json", OUTPUT / "data_manifest.json")
shutil.copy2(PROJECT / "configs/data_sources.json", OUTPUT / "data_sources.json")
shutil.copy2(PROJECT / "configs/massive_vi_mapping.json", OUTPUT / "massive_vi_mapping.json")
if (PROJECT / "benchmarks/holdout_v2.manifest.json").exists():
    shutil.copy2(PROJECT / "benchmarks/holdout_v2.manifest.json", OUTPUT / "holdout_v2.manifest.json")
print("Báo cáo chuẩn bị dữ liệu (Data Preparation Report):")
print(json.dumps(report, ensure_ascii=False, indent=2))


In [ ]:
if SMOKE:
    from pipeline_smoke import smoke
    smoke_res = smoke(OUTPUT / "smoke")
    CHECKPOINT = OUTPUT / "smoke/smoke.pt"
    print("CPU Smoke run và kiểm tra resume hoàn tất:", smoke_res)
else:
    from colab_helper import drive_config, BASELINE_SHA256
    from dusnx_core.training import train
    from dusnx_core.data_pipeline import sha256
    if BASELINE.exists():
        assert sha256(BASELINE) == BASELINE_SHA256, "Sai checkpoint baseline mốc 23aaa48"
    config = drive_config(PROJECT, OUTPUT)
    RESUME = False  # Đổi thành True nếu runtime Colab bị ngắt và bạn muốn tiếp tục từ router.last.pt
    last_checkpoint = OUTPUT / "router.last.pt"
    train(config, resume=last_checkpoint if RESUME and last_checkpoint.exists() else None,
          init_from=BASELINE if BASELINE.exists() and not RESUME else None)
    CHECKPOINT = OUTPUT / "router.pt"
    print("Huấn luyện thành công. Checkpoint tốt nhất:", CHECKPOINT)


## Đánh giá trung thực trên tập Holdout đã khóa
Đánh giá đồng thời 5 cấu hình độc lập: `baseline_a` (no memory), `baseline_b` (static memory), `model_only` (pure checkpoint, không luật), `dusnx_no_state` (ablation không recurrent state), và `dusnx` (full system).
Mặc định dùng `provider=mock` để kiểm tra pipeline logic trên Colab mà không phụ thuộc Ollama local.

In [ ]:
from colab_helper import evaluate_pair
eval_res = evaluate_pair(PROJECT, CHECKPOINT, BASELINE if BASELINE.exists() else None,
                         OUTPUT / "evaluation", provider="mock",
                         benchmark="benchmarks/holdout_v2.jsonl",
                         holdout_manifest="benchmarks/holdout_v2.manifest.json")
print("Trạng thái đánh giá:", json.dumps(eval_res, indent=2))
summary_file = OUTPUT / "evaluation/new/summary.md"
if summary_file.exists():
    print("\n" + summary_file.read_text(encoding="utf-8"))


In [ ]:
import shutil
destination = PROJECT / "artifacts" / RUN_NAME
shutil.copytree(OUTPUT, destination, dirs_exist_ok=True)
print("Đã lưu checkpoint, config, seed, hash data, nhãn và metric epoch tại:", OUTPUT)
print("Windows: mở Google Drive -> DUSNX -> Bấm phải thư mục", RUN_NAME, "-> Download; giải nén vào artifacts/.")
